# Guilin akustik rulman veri seti — indirme, deney, sonuçlar

Veri: *A multi-condition acoustic dataset of ball bearings for fault diagnosis* (Mendeley Data, 2026, CC BY 4.0).
5 durum (normal, dış bilezik, iç bilezik, kafes kırığı, bileşik iç-dış) × 9 koşul (800/1000/1200 rpm × %0/15/30 yük)
× 5 adet 10 dakikalık `.bkc` dosyası = 225 dosya, ~53 GB.

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. CPU'da CNN AE eğitimi saatler sürer.

Akış (toplam ~1 saat): depoyu al → (isteğe bağlı) normal/arıza karşılaştırması → 225 dosyayı indir ve 10 sn'lik WAV
kliplerine çevir (~12 dk) → koşul-dışı bırakma deneyi (log mel önbelleği ~15 dk + eğitimler) → grafikler → sonuç paketi.
Ham `.bkc` dosyaları saklanmaz; WAV'lar (~8,3 GB) Colab'ın geçici diskinde kalır.

In [ ]:
!nvidia-smi -L
!rm -rf /content/repo && git clone -q -b askinomer/inconnu https://github.com/askinomer/makine-ariza-tespit.git /content/repo
%cd /content/repo
!git log --oneline -1
!df -h /content | tail -1

## 1. (İsteğe bağlı) Normal ile arızalı kaydın karşılaştırılması — 1000 rpm, %0 yük, dosya 1

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
from pathlib import Path
from tools.guilin_convert import list_remote, download

cmp_dir = Path('/content/cmp'); cmp_dir.mkdir(exist_ok=True)
want = {'normal': 'nm1000r-0-32Khz.1.bkc', 'outer': 'Ods1000r-0-32Khz.1.bkc', 'inner': 'Ids1000r-0-32Khz.1.bkc',
        'cage': 'lc1000r-0-32Khz.1.bkc'}
paths = {}
for label, name in want.items():
    e = next(f for f in list_remote(label) if f['name'] == name)
    p = cmp_dir / name
    if not p.exists():
        download(e, p)
    paths[label] = p
!python tools/guilin_compare.py --normal {paths['normal']} --fault outer={paths['outer']} --fault inner={paths['inner']} --fault cage={paths['cage']} --rpm 1000 --out /content/cmp/guilin_compare.png
from IPython.display import Image, display
display(Image('/content/cmp/guilin_compare.png'))

## 2. Tüm veri setini indir ve WAV kliplerine çevir (~12 dk, kaldığı yerden devam eder)

In [ ]:
!python tools/guilin_convert.py --out /content/guilin_wav --workers 6 2>&1 | tail -8

## 3. Deney: koşul-dışı bırakma (9 kat), Isolation Forest ve CNN AE (3 tohum), gürültü, bootstrap, demo modelleri

Uzun sürdüğü için arka planda çalışır; ilerleme için bir sonraki hücreyi tekrar çalıştırın.

In [ ]:
!nohup python scripts/guilin_experiment.py --wav /content/guilin_wav --out /content/guilin_results --ckpt /content/guilin_ckpt --seeds 42 43 44 > /content/experiment.log 2>&1 &

In [ ]:
!tail -5 /content/experiment.log

## 4. Sonuçlar, grafikler ve sonuç paketi

In [ ]:
!python tools/make_guilin_figures.py --results /content/guilin_results
!cat /content/guilin_results/summary.md
from IPython.display import Image, display
for f in ('guilin_auc', 'guilin_kosul', 'guilin_gurultu'):
    display(Image(f'/content/guilin_results/figures/{f}.png'))

In [ ]:
# Repo düzeninde zip (results_guilin/, checkpoints_guilin/, app/examples/guilin/, data/guilin_examples.csv), ~9 MB
!python tools/guilin_bundle.py --results /content/guilin_results --ckpt /content/guilin_ckpt --wav /content/guilin_wav --out /content/guilin_sonuclar.zip
# from google.colab import files; files.download('/content/guilin_sonuclar.zip')   # bilgisayara indirmek için